In [1]:
import time
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
import mysql.connector

In [ ]:
# --- 訓練與評估 ---
def train_and_evaluate_models(X_train, X_test, y_train, y_test, scaler, test_dates, source, currency):
    results = []

    models = {
        "XGB": {"type": "xgboost", "params": {"n_estimators": 200, "max_depth": 5, "learning_rate": 0.05}},
        "LGBM": {"type": "lightgbm", "params": {"n_estimators": 200, "num_leaves": 31, "learning_rate": 0.05}},
        "CatBoost": {"type": "catboost", "params": {"iterations": 200, "depth": 6, "learning_rate": 0.05, "verbose": 0}}
    }

    for name, params in models.items():
        print(f"▶ 訓練 {name} 模型中...")

        start = time.time()

        if name == "XGB":
            model = xgb.XGBRegressor(**params["params"])
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test).reshape(-1, 1)

        elif name == "LGBM":
            model = lgb.LGBMRegressor(**params["params"])
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test).reshape(-1, 1)

        elif name == "CatBoost":
            model = CatBoostRegressor(**params["params"])
            model.fit(X_train, y_train, verbose=0)
            y_pred = model.predict(X_test).reshape(-1, 1)

        duration = time.time() - start

        # 反轉縮放
        y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))
        y_pred_inv = scaler.inverse_transform(y_pred)

        # 評估
        mae = mean_absolute_error(y_test_inv, y_pred_inv)
        rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
        mape = np.mean(np.abs((y_test_inv - y_pred_inv) / y_test_inv)) * 100
        r2 = r2_score(y_test_inv, y_pred_inv)

        print(f"✅ {name} 完成: MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}")

        results.append({
            "model": name,
            "source": source,
            "currency": currency,
            "params": params["params"],
            "mae": mae,
            "rmse": rmse,
            "mape": mape,
            "r2": r2,
            "time": duration
        })

    return pd.DataFrame(results)
